# Cleaning the Raw Data: California, 2016–2020

Builds `data/clean_ca_2016_2020_audited.parquet`, the shared input for the baselines and MLP.

In [ ]:
import sys; sys.path.append("..")
from pathlib import Path
import re
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from src.data import DATA_PATH, SPLIT_SEED, check_saved_split, cv_folds, feature_sets, split

## 1. Load the 5 yearly files, keeping only California fires

In [ ]:
expected_years = range(2016, 2021)
files = {}
for file in sorted(DATA_PATH.parent.glob("*_FPA_FOD_cons*.csv")):
    match = re.fullmatch(r"(\d{4})_FPA_FOD_cons(?: \(\d+\))?\.csv", file.name)
    if not match:
        continue
    year = int(match[1])
    if year not in expected_years:
        continue
    if year in files:
        raise ValueError(f"Multiple CSVs for {year}: {files[year].name}, {file.name}")
    files[year] = file

missing_years = sorted(set(expected_years) - files.keys())
if missing_years:
    raise FileNotFoundError(f"Missing raw CSVs for years: {missing_years}")

yearly = []
for year in expected_years:
    california = pd.read_csv(files[year], low_memory=False).query("STATE == 'CA'")
    if california.empty or not california["FIRE_YEAR"].eq(year).fillna(False).all():
        raise ValueError(f"{files[year].name} must contain California records for {year}")
    yearly.append(california)

df = pd.concat(yearly, ignore_index=True)
if df["FOD_ID"].isna().any() or df["FOD_ID"].duplicated().any():
    raise ValueError("Expected one record per non-missing FOD_ID")
df["FIRE_YEAR"].value_counts().sort_index()

## 2. First look

In [46]:
pd.set_option("display.max_rows", 400)
print(df["FIRE_SIZE_CLASS"].value_counts().sort_index())   # target balance
df.isna().mean().sort_values(ascending=False).head(30)     # most-missing columns

FIRE_SIZE_CLASS
A    27279
B    13678
C     1693
D      394
E      197
F      177
G      143
Name: count, dtype: int64


geometry                         1.000000
IALMIL_87                        1.000000
GACC_Fire Use Teams              1.000000
IAHSEF                           1.000000
IAULHS_89                        1.000000
IAPLHS_88                        1.000000
road_interstate_dis              0.999242
road_US_dis                      0.997681
ICS_209_PLUS_COMPLEX_JOIN_ID     0.994628
COMPLEX_NAME                     0.994582
MTBS_ID                          0.993274
MTBS_FIRE_NAME                   0.993274
road_other_dis                   0.993182
road_state_dis                   0.986410
ICS_209_PLUS_INCIDENT_JOIN_ID    0.983747
road_county_dis                  0.983586
NWCG_CAUSE_AGE_CATEGORY          0.973692
Evacuation                       0.915521
LOCAL_FIRE_REPORT_ID             0.844150
No_FireStation_1.0km             0.813801
FIRE_CODE                        0.803861
CONT_TIME                        0.576272
CONT_DATE                        0.550814
CONT_DOY                         0

## 3. Drop leakage and ID columns

- **Leakage:** anything only known after the fire (containment, cause, MTBS/ICS-209 links that only exist for large fires).
- **`_5D_` columns:** the 5-day windows are centered on ignition (2 days before to 2 days after), so they include post-ignition weather (Pourmohamad et al., 2024).
- `FIRE_SIZE` is kept for analysis only, never as a model input.

In [47]:
drop = [
    # Known only after the fire
    "CONT_DATE", "CONT_DOY", "CONT_TIME",
    "NWCG_CAUSE_CLASSIFICATION", "NWCG_GENERAL_CAUSE", "NWCG_CAUSE_AGE_CATEGORY",
    "MTBS_ID", "MTBS_FIRE_NAME", "ICS_209_PLUS_INCIDENT_JOIN_ID", "ICS_209_PLUS_COMPLEX_JOIN_ID",
    "COMPLEX_NAME",
    # IDs and names (no predictive meaning)
    "FOD_ID", "FPA_ID", "SOURCE_SYSTEM_TYPE", "SOURCE_SYSTEM", "NWCG_REPORTING_AGENCY",
    "NWCG_REPORTING_UNIT_ID", "NWCG_REPORTING_UNIT_NAME", "SOURCE_REPORTING_UNIT",
    "SOURCE_REPORTING_UNIT_NAME", "LOCAL_FIRE_REPORT_ID", "LOCAL_INCIDENT_ID",
    "FIRE_CODE", "FIRE_NAME",
]
drop += [c for c in df.columns if "_5D_" in c]   # post-ignition weather windows

df = df.drop(columns=drop, errors="ignore").copy()
print(df.shape)
print("Remaining _5D_ columns:", [c for c in df.columns if "_5D_" in c])   # should be []

(43561, 253)
Remaining _5D_ columns: []


## 4. Placeholder values and dates

In [ ]:
# Treat -999 / -9999 style placeholders as missing.
num_cols = df.select_dtypes("number").columns
df[num_cols] = df[num_cols].mask(df[num_cols] <= -999)
df = df.copy()

df["DISCOVERY_DATE"] = pd.to_datetime(df["DISCOVERY_DATE"], errors="coerce")
dates = df["DISCOVERY_DATE"].dt
if not (dates.year.eq(df["FIRE_YEAR"]) & dates.day_of_year.eq(df["DISCOVERY_DOY"])).all():
    raise ValueError("Discovery dates must match FIRE_YEAR and DISCOVERY_DOY")
df["DISCOVERY_MONTH"] = dates.month

# Values outside the physical range become missing for training-only imputation.
df["NDVI-1day"] = df["NDVI-1day"].where(df["NDVI-1day"].between(-1, 1))
for col in ("Slope", "Slope_1km"):
    if col in df:
        df[col] = df[col].where(df[col] >= 0)

## 5. Validate records and use training rows to screen missing columns

The regional split defines which records can determine the feature schema. Median imputation is fitted later using the same training rows.

In [ ]:
valid = (
    df["FIRE_SIZE_CLASS"].isin(list("ABCDEFG"))
    & np.isfinite(df["FIRE_SIZE"]) & (df["FIRE_SIZE"] >= 0)
    & df["LATITUDE"].between(-90, 90) & df["LONGITUDE"].between(-180, 180)
)
if not valid.fillna(False).all():
    raise ValueError(f"{(~valid.fillna(False)).sum()} records have invalid size, class, or coordinates")

# Whole 0.5° grid blocks are assigned to one partition.
lat_block = np.floor(df["LATITUDE"] / 0.5).astype(int).astype(str)
lon_block = np.floor(df["LONGITUDE"] / 0.5).astype(int).astype(str)
df["REGION"] = lat_block + "_" + lon_block
parts = split(df, seed=SPLIT_SEED)
print(check_saved_split(parts))
training_rows = parts[0].index

missing = df.loc[training_rows].isna().mean()
print(missing[missing > 0.5].sort_values(ascending=False))
df = df.loc[:, missing <= 0.5].copy()
df.shape

## 6. Target: merge the rarest size classes

In [50]:
merge = {"A": "A", "B": "B", "C": "C", "D": "D-E", "E": "D-E", "F": "F-G", "G": "F-G"}
df["TARGET"] = df["FIRE_SIZE_CLASS"].map(merge)
df["TARGET"].value_counts().sort_index()

/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/2609247854.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["TARGET"] = df["FIRE_SIZE_CLASS"].map(merge)


TARGET
A      27279
B      13678
C       1693
D-E      591
F-G      320
Name: count, dtype: int64

## 7. Convert weather percentile bands to numeric midpoints

Labels like `"<10th"`, `"10th-30th"`, and `">90th"` become the midpoint of the band. Both air-temperature percentile fields are excluded from the weather feature list because every raw California record reports `">90%"`.

In [51]:
pct_cols = [c for c in df.columns if c.endswith("_Percentile")]
print("Before:", df[pct_cols[0]].value_counts(dropna=False).to_dict())

def pct_to_number(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.number)):
        return float(v)
    s = str(v).strip()
    nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", s)]
    if not nums:
        return np.nan
    if s.startswith("<"):
        return nums[0] / 2              # "<10th"  -> 5
    if s.startswith(">"):
        return (nums[0] + 100) / 2      # ">90th"  -> 95
    return sum(nums) / len(nums)        # "10th-30th" -> 20

for c in pct_cols:
    df[c] = df[c].map(pct_to_number)

print("After:", df[pct_cols[0]].value_counts(dropna=False).sort_index().to_dict())

Before: {'>90%': 43561}
After: {95.0: 43561}


In [ ]:
# 7b. Summarize monthly NDVI, preserving missing month positions.
def parse_ndvi_months(value):
    if pd.isna(value):
        return [np.nan] * 12
    tokens = [token for token in re.split(r"""[\s,\[\]'"]+""", str(value)) if token]
    if len(tokens) != 12:
        raise ValueError(f"Expected 12 monthly NDVI values, got {len(tokens)}")
    return [float(token) for token in tokens]

for col in ("NDVI_min", "NDVI_max", "NDVI_mean", "MOD_NDVI_12m"):
    months = pd.DataFrame(df[col].map(parse_ndvi_months).tolist(), index=df.index)
    months = months.where(months.abs() <= 1)
    df[col + "_avg"] = months.mean(axis=1)
    # The source lists months from most to least recent. A missing first month stays missing.
    df[col + "_last"] = months.iloc[:, 0]
    df = df.drop(columns=col)

print(df[[col for col in df.columns if col.endswith(("_avg", "_last"))]].describe().round(3))

## 8. Category codes → one-hot columns

Keep the 20 most common land-cover and vegetation codes **in the training regions**. Map other codes to `other` and missing values to `missing`.

In [ ]:
cat_cols = [c for c in ["Land_Cover", "EVT"] if c in df.columns]
for c in cat_cols:
    codes = df[c]
    top = codes.loc[training_rows].value_counts().head(20).index
    df[c] = codes.astype("string").where(codes.isin(top) | codes.isna(), "other").fillna("missing")

df = pd.get_dummies(df, columns=cat_cols, prefix=["LC" if c == "Land_Cover" else "EVT" for c in cat_cols], dtype=float)
print([c for c in feature_sets(df)["model2_weather_veg"] if c.startswith(("LC_", "EVT_"))])

## 9. Aspect is a compass direction → sine and cosine

**Aspect → sin/cos:** Aspect is a compass direction (0°–360°), and 359° and 1° both face north even though the numbers are far apart, so we convert it to `cos` (north = +1, south = −1) and `sin` (east = +1, west = −1). That way the model sees nearby directions as similar.

In [54]:
for c, s in [("Aspect", "Slope"), ("Aspect_1km", "Slope_1km")]:
    if c in df.columns:
        rad = np.deg2rad(df[c])
        df[c + "_sin"], df[c + "_cos"] = np.sin(rad), np.cos(rad)
        if s in df.columns:
            flat = df[s] < 2                      # nearly flat: no real facing direction
            df.loc[flat, [c + "_sin", c + "_cos"]] = 0

/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/4032990377.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c + "_sin"], df[c + "_cos"] = np.sin(rad), np.cos(rad)
/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/4032990377.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[c + "_sin"], df[c + "_cos"] = np.sin(rad), np.cos(rad)
/var/folders/w2/4nk24xz14yg09dbpyrxdn4c80000gn/T/ipykernel_45106/4032990377.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result 

**Flat ground fix:** Flat ground doesn't face any direction, but the data records it as 0°, which would be read as "north-facing." During EDA this made north-facing slopes look much safer than they really are, because flat areas like farms and towns rarely have large fires. So for nearly flat ground (slope < 2°), we set both `sin` and `cos` to 0, meaning "no direction."

## 10. Review the regional distribution of large fires

In [ ]:
print(df["REGION"].nunique(), "regions")
fg = df.groupby("REGION")["TARGET"].apply(lambda s: (s == "F-G").sum()).sort_values(ascending=False)
print((fg > 0).sum(), "regions contain at least one F-G fire")
fg.head(10)

## 11. Check the feature groups and the folds

In [ ]:
sets = feature_sets(df)
for name, cols in sets.items():
    print(name, len(cols), "features")

# Class mix in each test fold (watch the F-G counts)
pd.DataFrame({f"fold {i}": te["TARGET"].value_counts()
              for i, (tr, te) in enumerate(cv_folds(df))}).sort_index()

## 12. Keep only what's needed and save

In [ ]:
extra = ["TARGET", "REGION", "FIRE_SIZE", "FIRE_SIZE_CLASS", "FIRE_YEAR", "STATE",
         "DISCOVERY_DATE", "Population", "GAP_Prity", "EPL_MOBILE"]
keep = list(dict.fromkeys(sets["model3_all"] + [c for c in extra if c in df.columns]))
df = df[keep].copy()

with TemporaryDirectory(dir=DATA_PATH.parent) as directory:
    pending_path = Path(directory) / DATA_PATH.name
    df.to_parquet(pending_path, index=False)
    check = pd.read_parquet(pending_path)
    pd.testing.assert_frame_equal(df.reset_index(drop=True), check)
    pending_path.replace(DATA_PATH)
print(DATA_PATH.name, check.shape)

## 13. Sanity check: reload the saved file

In [ ]:
from src.data import load

check = load()
print(check.shape)
check["TARGET"].value_counts().sort_index()